In [18]:
import torch
from torch.nn import functional as F
from kerops.ops.linear.linear_bias_relu_linear_add import LinBReLULinAdd, autotune_lin_bn_relu_lin_add, generate_inputs_lin_bn_relu_lin_add
from kerops.ops.assets import ASSETS_ROOT

In [2]:
autotune_lin_bn_relu_lin_add(ASSETS_ROOT / 'LinBReLULinAdd.toml', n_jobs_precompile=4)

Problem sizes:   0%|          | 0/3 [00:00<?, ?it/s]

Precompiling:   0%|          | 0/36 [00:00<?, ?it/s]

Benchmark configs:   0%|          | 0/36 [00:00<?, ?it/s]

Precompiling:   0%|          | 0/24 [00:00<?, ?it/s]

Benchmark configs:   0%|          | 0/24 [00:00<?, ?it/s]

Precompiling:   0%|          | 0/24 [00:00<?, ?it/s]

Benchmark configs:   0%|          | 0/24 [00:00<?, ?it/s]

In [70]:
channels = 64

x, weight_up, weight_down, bias, add_other = generate_inputs_lin_bn_relu_lin_add({'in_channels': channels})

In [72]:
%%timeit -r 10 -n 10
LinBReLULinAdd(x, weight_up, weight_down, bias, add_other)
torch.cuda.synchronize()

636 μs ± 43.6 μs per loop (mean ± std. dev. of 10 runs, 10 loops each)


In [74]:
%%timeit -r 10 -n 10
with torch.amp.autocast('cuda'), torch.inference_mode():
    o = F.linear(x.permute(0, 2, 3, 4, 1), weight_up.T, bias)
    o = F.relu(o)
    o = F.linear(o, weight_down.T, None)
    o += add_other.permute(0, 2, 3, 4, 1)
torch.cuda.synchronize()

2.62 ms ± 89.6 μs per loop (mean ± std. dev. of 10 runs, 10 loops each)
